# 2B ablation — re-run of the two unfinished experiments (wd_on_bn / no_label_smoothing)

In the main notebook these two runs were lost when the Kaggle session was recycled, so they are re-run here on their own.
Environment setup, data, training framework and the ResNet-18 builder are identical to the main notebook; everything else is left out.
**Usage**: in Kaggle turn on GPU and Internet in Settings, then *Save Version → Save & Run All (Commit)*; expected run time ≈ 2.5 h.

## Part 0 — Environment and configuration

The environment is detected automatically (Colab / Kaggle / SLURM cluster / local) and sets `DATA_ROOT`, `DEVICE` and the random seed.

**Kaggle notes**
- Turn on `Settings → Internet: On` (otherwise the dataset download fails; enabling GPU + internet for the first time requires phone verification of the account).
- Use an interactive **Edit** session for debugging (like Colab, it can be recycled when the browser disconnects). For long unattended runs use **Save & Run All (Commit)**: the whole notebook is executed on Kaggle's servers independently of your browser. A single commit run is capped at about 12 hours.
- Checkpoints and results are written to `/kaggle/working/` and are kept with the version output.

**Cluster (SLURM) notes**
- `$HOME` has a 400 MB quota, so the data and checkpoints must go to scratch space. The environment variables `IPCV_DATA_ROOT` and `IPCV_CKPT_DIR` override the defaults on any platform (see `cluster/run_notebook.sbatch`).

In [ ]:
import os, sys, math, time, json, random, collections
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from torchvision.datasets import FGVCAircraft
import matplotlib.pyplot as plt
from IPython.display import display   # also available when the notebook is run as a plain script

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_SLURM  = "SLURM_JOB_ID" in os.environ

# Where the dataset lives. IPCV_DATA_ROOT (if set) always wins. On the DISI cluster $HOME has a
# 400 MB quota, so there it must point to /scratch.hpc/... (see cluster/run_notebook.sbatch).
if os.environ.get("IPCV_DATA_ROOT"):
    DATA_ROOT = Path(os.environ["IPCV_DATA_ROOT"])
elif IN_COLAB:
    DATA_ROOT = Path("/content/data")
elif IN_KAGGLE:
    DATA_ROOT = Path("/kaggle/working/data")
else:
    DATA_ROOT = Path.home() / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()
torch.backends.cudnn.benchmark = True

PLATFORM = ("Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE
            else "SLURM cluster" if IN_SLURM else "local")
print(f"torch {torch.__version__} | tv {torchvision.__version__}")
print(f"platform={PLATFORM} | device={DEVICE} | data_root={DATA_ROOT}")
if IN_KAGGLE:
    print("Kaggle reminder: make sure Settings -> Internet is ON, otherwise the dataset download fails.")

In [ ]:
# --- Robust FGVC-Aircraft download (the Oxford VGG site sometimes returns 403 / times out) ---
# torchvision expects <DATA_ROOT>/fgvc-aircraft-2013b/ ; if the tarball already exists it skips the download and only extracts.
import urllib.request, shutil, ssl

_TAR = DATA_ROOT / "fgvc-aircraft-2013b.tar.gz"
_URL = "https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/archives/fgvc-aircraft-2013b.tar.gz"

if (DATA_ROOT / "fgvc-aircraft-2013b").exists():
    print("Already extracted, skipping")
elif _TAR.exists() and _TAR.stat().st_size > 2_000_000_000:
    print(f"Tarball in place ({_TAR.stat().st_size/1e9:.2f} GB), letting torchvision extract it")
else:
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    for attempt in range(1, 6):
        try:
            print(f"Download attempt {attempt}/5 ...")
            req = urllib.request.Request(_URL, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120, context=ctx) as r, open(_TAR, "wb") as f:
                shutil.copyfileobj(r, f, length=1 << 20)
            print(f"Done: {_TAR.stat().st_size/1e9:.2f} GB")
            break
        except Exception as e:
            print("  failed:", e)
    else:
        print("\nAutomatic download failed. Manual fallback: upload fgvc-aircraft-2013b.tar.gz to\n "
              f"{_TAR}\nthen re-run this cell and the next one (torchvision extracts it automatically).")

In [ ]:
NUM_CLASSES = 100

def load_raw(split):
    return FGVCAircraft(root=str(DATA_ROOT), split=split,
                        annotation_level="variant", download=True, transform=None)

raw_train, raw_val, raw_test = load_raw("train"), load_raw("val"), load_raw("test")
print(f"train {len(raw_train)} | val {len(raw_val)} | test {len(raw_test)}")
print(f"#classes = {len(raw_train.classes)}  e.g. {raw_train.classes[:4]}")
assert len(raw_train.classes) == NUM_CLASSES

In [ ]:
class CropBottomBanner:
    """Remove the ~20 px copyright banner at the bottom of every FGVC-Aircraft image"""
    def __init__(self, px=20): self.px = px
    def __call__(self, img):
        w, h = img.size
        return img.crop((0, 0, w, h - self.px))

IMG_SIZE  = 128
RESIZE_TO = int(round(IMG_SIZE * 1.14))   # resize first, then center-crop, to keep the aspect ratio

def compute_mean_std(base, n=1500):
    tf = transforms.Compose([CropBottomBanner(20),
                             transforms.Resize(RESIZE_TO),
                             transforms.CenterCrop(IMG_SIZE),
                             transforms.ToTensor()])
    idx = np.random.default_rng(SEED).choice(len(base), min(n, len(base)), replace=False)
    mean = torch.zeros(3); sq = torch.zeros(3)
    for i in idx:
        x = tf(base[i][0]); mean += x.mean((1, 2)); sq += (x**2).mean((1, 2))
    mean /= len(idx); std = (sq/len(idx) - mean**2).sqrt()
    return mean.tolist(), std.tolist()

FGVC_MEAN, FGVC_STD = compute_mean_std(raw_train)
print("mean", [round(v, 4) for v in FGVC_MEAN], "std", [round(v, 4) for v in FGVC_STD])

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_train_tf(img_size=IMG_SIZE, mean=None, std=None,
                   augment=True, rrc=True, hflip=True, jitter=True):
    mean = mean or FGVC_MEAN; std = std or FGVC_STD
    steps = [CropBottomBanner(20)]
    if augment and rrc:
        steps.append(transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0), ratio=(0.8, 1.25)))
    else:
        steps += [transforms.Resize(int(round(img_size*1.14))), transforms.CenterCrop(img_size)]
    if augment and hflip:  steps.append(transforms.RandomHorizontalFlip())
    if augment and jitter: steps.append(transforms.ColorJitter(0.2, 0.2, 0.2))
    steps += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    return transforms.Compose(steps)

def build_eval_tf(img_size=IMG_SIZE, mean=None, std=None):
    mean = mean or FGVC_MEAN; std = std or FGVC_STD
    return transforms.Compose([CropBottomBanner(20),
                               transforms.Resize(int(round(img_size*1.14))),
                               transforms.CenterCrop(img_size),
                               transforms.ToTensor(), transforms.Normalize(mean, std)])

In [ ]:
class Transformed(Dataset):
    def __init__(self, base, tf): self.base, self.tf = base, tf
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, y = self.base[i]; return self.tf(img), y

def make_loaders(img_size=IMG_SIZE, batch_size=128, mean=None, std=None,
                 augment=True, rrc=True, hflip=True, jitter=True, num_workers=2):
    ttf = build_train_tf(img_size, mean, std, augment, rrc, hflip, jitter)
    etf = build_eval_tf(img_size, mean, std)
    common = dict(num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"))
    return (DataLoader(Transformed(raw_train, ttf), batch_size, shuffle=True,  drop_last=True, **common),
            DataLoader(Transformed(raw_val,   etf), batch_size, shuffle=False,                 **common),
            DataLoader(Transformed(raw_test,  etf), batch_size, shuffle=False,                 **common))

In [ ]:
from dataclasses import dataclass, asdict

ACT = {"relu": nn.ReLU, "gelu": nn.GELU, "leaky_relu": lambda: nn.LeakyReLU(0.1)}

@dataclass
class NetConfig:
    num_classes:      int   = 100
    stem_channels:    int   = 32
    stage_channels:   tuple = (64, 128, 256, 512)
    blocks_per_stage: int   = 2
    use_bn:           bool  = True         # ablation
    use_residual:     bool  = True         # ablation
    activation:       str   = "relu"       # relu | gelu | leaky_relu   ablation
    global_pool:      str   = "avg"        # "avg" | "flatten"          ablation
    dropout:          float = 0.3          # ablation
    in_size:          int   = 128

def _norm(c, use_bn):
    return nn.BatchNorm2d(c) if use_bn else nn.Identity()

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride, cfg):
        super().__init__()
        act = ACT[cfg.activation]
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=not cfg.use_bn)
        self.n1, self.act1 = _norm(cout, cfg.use_bn), act()
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=not cfg.use_bn)
        self.n2, self.act2 = _norm(cout, cfg.use_bn), act()
        self.use_residual = cfg.use_residual
        if cfg.use_residual and (stride != 1 or cin != cout):
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=not cfg.use_bn),
                                       _norm(cout, cfg.use_bn))
        else:
            self.short = nn.Identity()

    def forward(self, x):
        out = self.act1(self.n1(self.conv1(x)))
        out = self.n2(self.conv2(out))
        if self.use_residual:
            out = out + self.short(x)
        return self.act2(out)

class AircraftNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        act = ACT[cfg.activation]
        self.stem = nn.Sequential(
            nn.Conv2d(3, cfg.stem_channels, 3, 2, 1, bias=not cfg.use_bn),
            _norm(cfg.stem_channels, cfg.use_bn), act())
        blocks, cin = [], cfg.stem_channels
        for cout in cfg.stage_channels:
            for bi in range(cfg.blocks_per_stage):
                blocks.append(BasicBlock(cin, cout, 2 if bi == 0 else 1, cfg))
                cin = cout
        self.stages = nn.Sequential(*blocks)
        feat = cfg.stage_channels[-1]
        if cfg.global_pool == "avg":
            self.pool, head_in = nn.AdaptiveAvgPool2d(1), feat
        elif cfg.global_pool == "flatten":
            r = cfg.in_size // 2 // (2 ** len(cfg.stage_channels))
            self.pool, head_in = nn.Identity(), feat * r * r
        else:
            raise ValueError(cfg.global_pool)
        self.drop = nn.Dropout(cfg.dropout)
        self.fc   = nn.Linear(head_in, cfg.num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x); x = self.stages(x); x = self.pool(x)
        x = torch.flatten(x, 1); x = self.drop(x)
        return self.fc(x)

def build_model(cfg=None):
    return AircraftNet(cfg or NetConfig())

_cfg = NetConfig(); _m = build_model(_cfg)
print(f"params: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M | "
      f"out: {tuple(_m(torch.randn(2, 3, _cfg.in_size, _cfg.in_size)).shape)}")
del _cfg, _m

In [ ]:
# Checkpoint / results directory:
#   IPCV_CKPT_DIR (env var) -> explicit override (cluster: a directory under /scratch.hpc)
#   Colab                   -> Google Drive (survives disconnects)
#   Kaggle                  -> /kaggle/working (kept with the version output, survives a background commit run)
#   otherwise               -> ./ckpts
if os.environ.get("IPCV_CKPT_DIR"):
    CKPT_DIR = Path(os.environ["IPCV_CKPT_DIR"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = Path("/content/drive/MyDrive/ipcv_assignment2/ckpts")
elif IN_KAGGLE:
    CKPT_DIR = Path("/kaggle/working/ckpts")
else:
    CKPT_DIR = Path("ckpts")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_JSON = CKPT_DIR.parent / "results.json"
print("CKPT_DIR    :", CKPT_DIR)
print("RESULTS_JSON:", RESULTS_JSON)

In [ ]:
from dataclasses import dataclass, field, asdict, replace

@dataclass
class TrainConfig:
    epochs:          int   = 80
    batch_size:      int   = 128
    optimizer:       str   = "sgd"       # sgd | adamw
    lr:              float = 0.1         # use ~3e-4 with adamw
    momentum:        float = 0.9
    weight_decay:    float = 5e-4
    nesterov:        bool  = True
    scheduler:       str   = "cosine"    # cosine | constant   <- ablation
    warmup_epochs:   int   = 5
    label_smoothing: float = 0.0         # Part 2B tuning
    no_wd_on_bn:     bool  = False       # Part 2B tuning: no weight decay on BN parameters / biases
    grad_clip:       float = 0.0
    amp:             bool  = True        # only effective on CUDA
    img_size:        int   = 128
    mean: tuple = None                   # None -> FGVC statistics computed on train; Part 2 passes IMAGENET_MEAN
    std:  tuple = None
    augment: bool = True
    rrc:     bool = True
    hflip:   bool = True
    jitter:  bool = True
    num_workers: int = int(os.environ.get("IPCV_NUM_WORKERS", 2))   # set IPCV_NUM_WORKERS on machines with more CPU cores
    seed: int = 42


def build_optimizer(model, cfg: TrainConfig):
    if cfg.no_wd_on_bn:
        # 1-D parameters = BN gamma/beta + all biases: no weight decay
        # (as in the torchvision 2021 re-training recipe presented in lecture P2L6)
        decay, no_decay = [], []
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            (no_decay if p.ndim <= 1 else decay).append(p)
        params = [{"params": decay, "weight_decay": cfg.weight_decay},
                  {"params": no_decay, "weight_decay": 0.0}]
    else:
        params = model.parameters()
    if cfg.optimizer == "sgd":
        return torch.optim.SGD(params, lr=cfg.lr, momentum=cfg.momentum,
                               weight_decay=cfg.weight_decay, nesterov=cfg.nesterov)
    if cfg.optimizer == "adamw":
        return torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    raise ValueError(cfg.optimizer)


def build_scheduler(opt, cfg: TrainConfig):
    """Stepped once per epoch. constant = flat LR (used by the "no schedule" ablation)."""
    if cfg.scheduler == "constant":
        return torch.optim.lr_scheduler.LambdaLR(opt, lambda _: 1.0)
    warm = torch.optim.lr_scheduler.LinearLR(
        opt, start_factor=1e-2, total_iters=max(1, cfg.warmup_epochs))
    cos = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(1, cfg.epochs - cfg.warmup_epochs))
    return torch.optim.lr_scheduler.SequentialLR(
        opt, [warm, cos], milestones=[max(1, cfg.warmup_epochs)])

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tot = 0; loss_sum = 0.0; c1 = 0; c5 = 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        out = model(x)
        loss_sum += criterion(out, y).item() * x.size(0); tot += x.size(0)
        top5 = out.topk(5, dim=1).indices
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
    return loss_sum / tot, 100 * c1 / tot, 100 * c5 / tot


def train_loop(model, train_loader, val_loader, cfg: TrainConfig, verbose=True):
    crit  = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    opt   = build_optimizer(model, cfg)
    sched = build_scheduler(opt, cfg)
    use_amp = cfg.amp and DEVICE.type == "cuda"
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    keys = ["lr", "train_loss", "train_top1", "val_loss", "val_top1", "val_top5"]
    hist = {k: [] for k in keys}
    best_val = -1.0; best_state = None

    for ep in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time(); seen = 0; run_loss = 0.0; run_c1 = 0
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward()
            if cfg.grad_clip > 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update()
            run_loss += loss.item() * x.size(0); seen += x.size(0)
            run_c1   += (out.argmax(1) == y).sum().item()
        sched.step()

        tr_loss = run_loss / seen; tr_top1 = 100 * run_c1 / seen
        va_loss, va_top1, va_top5 = evaluate(model, val_loader, crit)
        lr_now = opt.param_groups[0]["lr"]
        for k, v in zip(keys, [lr_now, tr_loss, tr_top1, va_loss, va_top1, va_top5]):
            hist[k].append(v)
        if va_top1 > best_val:
            best_val = va_top1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"ep {ep:3d}/{cfg.epochs} | lr {lr_now:.4f} | "
                  f"train {tr_loss:.3f}/{tr_top1:5.1f}% | "
                  f"val {va_loss:.3f}/{va_top1:5.1f}% (top5 {va_top5:4.1f}%) | "
                  f"{time.time() - t0:4.1f}s")
    return hist, best_state, best_val

In [ ]:
RESULTS = {}
# Re-load results written by earlier sessions (e.g. after a kernel restart), so that tables and plots
# can be rebuilt without re-training.
if RESULTS_JSON.exists():
    try:
        RESULTS.update(json.loads(RESULTS_JSON.read_text())["full"])
        print(f"Re-loaded {len(RESULTS)} earlier results from {RESULTS_JSON}")
    except Exception as e:
        print("Could not re-load", RESULTS_JSON, "->", e)

def _dump_results():
    slim = {k: {kk: vv for kk, vv in v.items() if kk != "history"} for k, v in RESULTS.items()}
    RESULTS_JSON.write_text(json.dumps({"slim": slim, "full": RESULTS}, indent=2, default=str))

def run_experiment(name, net_cfg=None, train_cfg=None, save_ckpt=True, verbose=True,
                   model_builder=None):
    """model_builder: optional custom model factory (Part 2 passes the ResNet-18 here); when given it
    bypasses build_model(net_cfg) and net_cfg is only kept for the record.
    With IPCV_SKIP_DONE=1 an experiment that is already stored in results.json is not re-trained
    (lets a long one-shot job be resumed after a crash / time-out)."""
    train_cfg = train_cfg or TrainConfig()
    if os.environ.get("IPCV_SKIP_DONE") == "1" and RESULTS.get(name, {}).get("history"):
        print(f"[{name}] already finished (found in results.json) -> skipped")
        return RESULTS[name]
    seed_everything(train_cfg.seed)
    tl, vl, testl = make_loaders(
        img_size=train_cfg.img_size, batch_size=train_cfg.batch_size,
        mean=train_cfg.mean, std=train_cfg.std,
        augment=train_cfg.augment, rrc=train_cfg.rrc,
        hflip=train_cfg.hflip, jitter=train_cfg.jitter, num_workers=train_cfg.num_workers)
    if model_builder is not None:
        model = model_builder().to(DEVICE)
        net_cfg_dict = {"builder": getattr(model_builder, "__name__", str(model_builder))}
    else:
        net_cfg = replace(net_cfg or NetConfig(), in_size=train_cfg.img_size)
        model = build_model(net_cfg).to(DEVICE)
        net_cfg_dict = asdict(net_cfg)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"=== {name} === {n_params/1e6:.2f}M params | img {train_cfg.img_size} | "
          f"{train_cfg.optimizer} lr{train_cfg.lr} wd{train_cfg.weight_decay} | "
          f"sched={train_cfg.scheduler} | {train_cfg.epochs} ep")
    t0 = time.time()
    hist, best_state, best_val = train_loop(model, tl, vl, train_cfg, verbose=verbose)
    model.load_state_dict(best_state)
    _, te1, te5 = evaluate(model, testl, nn.CrossEntropyLoss())
    rec = dict(name=name, test_top1=te1, test_top5=te5, best_val_top1=best_val,
               n_params=int(n_params), minutes=(time.time() - t0) / 60,
               history=hist, net_cfg=net_cfg_dict, train_cfg=asdict(train_cfg))
    RESULTS[name] = rec
    if save_ckpt:
        torch.save({"model": best_state, "net_cfg": rec["net_cfg"],
                    "train_cfg": asdict(train_cfg), "test_top1": te1},
                   CKPT_DIR / f"{name}.pt")
    _dump_results()
    print(f"[{name}]  test top1 {te1:.2f}%  top5 {te5:.2f}%  |  best val {best_val:.2f}%  "
          f"|  {rec['minutes']:.1f} min")
    return rec


def plot_history(hist_or_name, title=""):
    h = RESULTS[hist_or_name]["history"] if isinstance(hist_or_name, str) else hist_or_name
    ep = range(1, len(h["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    ax[0].plot(ep, h["train_loss"], label="train"); ax[0].plot(ep, h["val_loss"], label="val")
    ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep, h["train_top1"], label="train"); ax[1].plot(ep, h["val_top1"], label="val")
    ax[1].set_title("top-1 (%)"); ax[1].set_xlabel("epoch"); ax[1].legend()
    ax[2].plot(ep, h["lr"]); ax[2].set_title("lr"); ax[2].set_xlabel("epoch")
    fig.suptitle(title or (hist_or_name if isinstance(hist_or_name, str) else ""))
    plt.tight_layout(); plt.show()


def results_table(names=None):
    import pandas as pd
    names = names or list(RESULTS)
    rows = [dict(experiment=n,
                 test_top1=round(RESULTS[n]["test_top1"], 2),
                 test_top5=round(RESULTS[n]["test_top5"], 2),
                 val_top1=round(RESULTS[n]["best_val_top1"], 2),
                 params_M=round(RESULTS[n]["n_params"] / 1e6, 2),
                 min=round(RESULTS[n]["minutes"], 1)) for n in names]
    return pd.DataFrame(rows).set_index("experiment")

In [ ]:
def build_resnet18(num_classes=100, pretrained=True, freeze_backbone=False):
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad_(False)
    model.fc = nn.Linear(model.fc.in_features, num_classes)  # the new head is always trainable
    return model

_m = build_resnet18()
print(f"resnet18 params: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M "
      f"(trainable: {sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.2f}M)")
del _m

In [ ]:
# Results of the runs that had already finished in an earlier session (numbers copied from their logs).
# They are only used for the final table / bar chart and are NOT re-trained.
def _stub(name, t1, t5, val, minutes):
    RESULTS[name] = dict(name=name, test_top1=t1, test_top5=t5, best_val_top1=val,
                         n_params=11227812, minutes=minutes, history=None,
                         note="carried over from an earlier session")
_stub("p2a_resnet18",       57.48574857485749,  83.73837383738373, 57.90579057905791, 58.634676325321195)
_stub("p2b_resnet18_tuned", 71.4971497149715,   91.14911491149115, 69.99699969996999, 70.08717394272486)
_stub("p2b_abl_128px",      59.045904590459045, 82.71827182718272, 58.92589258925893, 59.14510685602824)
print(list(RESULTS))

In [ ]:
# --- 2B ablation 2: remove no_wd_on_bn ---
run_experiment(
    "p2b_abl_wd_on_bn",
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3, img_size=224,
                          mean=IMAGENET_MEAN, std=IMAGENET_STD,
                          no_wd_on_bn=False, label_smoothing=0.1),
    model_builder=lambda: build_resnet18(num_classes=NUM_CLASSES),
)

In [ ]:
# --- 2B ablation 3: remove label smoothing ---
run_experiment(
    "p2b_abl_no_label_smoothing",
    train_cfg=TrainConfig(epochs=80, weight_decay=1e-3, img_size=224,
                          mean=IMAGENET_MEAN, std=IMAGENET_STD,
                          no_wd_on_bn=True, label_smoothing=0.0),
    model_builder=lambda: build_resnet18(num_classes=NUM_CLASSES),
)

In [ ]:
P2B_ABLATION_NAMES = ["p2a_resnet18", "p2b_abl_128px", "p2b_abl_wd_on_bn",
                      "p2b_abl_no_label_smoothing", "p2b_resnet18_tuned"]
tbl = results_table(P2B_ABLATION_NAMES)
display(tbl)

fig, ax = plt.subplots(figsize=(10, 4))
vals = [RESULTS[n]["test_top1"] for n in P2B_ABLATION_NAMES]
colors_ = ["#999999", "#c0392b", "#c0392b", "#c0392b", "#2f5da8"]
ax.bar(range(len(P2B_ABLATION_NAMES)), vals, color=colors_)
ax.set_xticks(range(len(P2B_ABLATION_NAMES)))
ax.set_xticklabels(["2A\n(no changes)", "-224px", "-no_wd_on_bn", "-label_smooth", "2B\n(all three)"],
                   rotation=15, ha="right")
ax.set_ylabel("test top-1 (%)")
ax.axhline(vals[-1], color="#2f5da8", linestyle="--", linewidth=1, alpha=0.6)
ax.set_title("2B ablation — which of the three changes contributes most?")
plt.tight_layout(); plt.show()